## pkcore.py Intro
A tour of `pkcore.py` — the Python bindings for the `pkcore` Rust poker engine.

Install name: `pkcore.py`  |  Import name: `pkcore`

In [ ]:
import pkcore

### Cards
Parse individual cards and inspect their rank and suit.

In [ ]:
c = pkcore.Card.parse("As")
print(f"Card   : {c}")
print(f"Rank   : {c.rank()} (value={c.rank().value()})")
print(f"Suit   : {c.suit()} (symbol={c.suit().symbol()})")
print(f"Cactus Kev u32: {c.as_u32():#010x}")

### Deck
Build a full 52-card deck and inspect it.

In [ ]:
deck = pkcore.Cards.deck()
print(f"Deck size: {len(deck)}")
print(f"Cards    : {deck}")

### Board
Parse community cards and see what's visible at each street.

In [ ]:
board = pkcore.Board.parse("As Ks Qh Jd Tc")
print(f"Full board  : {board}")
print(f"Through turn: {board.turn_cards()}")

### Combo ranges
A `Combo` is an abstract hand category like `AKs` or `QQ+`.
`Combos` is a range; `.explode()` expands it into every concrete two-card hand.

In [ ]:
# Top ~2.5% of starting hands: QQ+, AK
range_str = pkcore.Combos.PERCENT_2_5
combos = pkcore.Combos.parse(range_str)
twos = combos.explode()

print(f"Range         : {range_str}")
print(f"Abstract combos: {len(combos)}")
print(f"Concrete hands : {len(twos)}")
print()

suited   = twos.filter_is_suited()
offsuit  = twos.filter_is_not_suited().filter_is_not_paired()
pairs    = twos.filter_is_paired()

print(f"  Suited  : {len(suited)}")
print(f"  Offsuit : {len(offsuit)}")
print(f"  Pairs   : {len(pairs)}")

### Outs calculation
Given hole cards and a turn board, find which river cards give each player the win.

In [ ]:
# Player 1: A♠ K♥  |  Player 2: 8♦ K♣  (top pair vs. two pair draw)
# Board through the turn: A♣ 8♥ 7♥ 9♠
hc    = pkcore.HoleCards.parse("As Kh 8d Kc")
board = pkcore.Board.parse("Ac 8h 7h 9s")
game  = pkcore.Game(hc, board)

case_evals = game.turn_case_evals()
outs = pkcore.Outs.from_case_evals(case_evals)

print(f"Possible river cards : {len(case_evals)}")
print()
for player in (1, 2):
    n = outs.len_for_player(player)
    cards = outs.get(player)
    winner = " <-- most outs" if outs.is_longest(player) else ""
    print(f"Player {player}: {n} outs — {cards}{winner}")

### Hand space stats
Quick look at the numbers behind Texas Hold'em.

In [ ]:
print(f"Unique 2-card starting hands : {pkcore.unique_2_card_hands():,}")
print(f"Distinct 2-card hand types   : {pkcore.distinct_2_card_hands():,}")
print(f"Unique 5-card hands (52C5)   : {pkcore.unique_5_card_hands():,}")
print(f"Distinct 5-card hand rankings: {pkcore.distinct_5_card_hands():,}")